In [0]:
# Databricks notebook source
# ==============================================================================
# TrackPulse — Data Quality Gate & Audit Logging
# Notebook: 02_quality_gate_prod
# Layer: Silver -> Quality Gate Evaluation -> Gold
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

# ------------------------------------------------------------------------------
# 1. Retrieve Parameters passed dynamically from Azure Data Factory (ADF)
# ------------------------------------------------------------------------------
dbutils.widgets.text("run_id", "manual_run", "Pipeline Run ID")
run_id = dbutils.widgets.get("run_id")

print(f"Executing Data Quality Gate for Pipeline Run ID: {run_id}")

# ------------------------------------------------------------------------------
# 2. Define Storage and Table Paths
# ------------------------------------------------------------------------------
storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
silver_paths = {
    "telemetry": storage_path + "silver/telemetry/",
    "trips": storage_path + "silver/trips/",
    "trip_plans": storage_path + "silver/master_data/trip_plans/",
    "vehicles": storage_path + "silver/master_data/vehicles/",
    "drivers": storage_path + "silver/master_data/drivers/",
    "routes": storage_path + "silver/master_data/routes/",
    "shipments": storage_path + "silver/master_data/shipments/"
}
audit_path = storage_path + "audit/data_quality_results/"

# ------------------------------------------------------------------------------
# 3. Load Validated Silver Datasets
# ------------------------------------------------------------------------------
telemetry_df = spark.read.format("delta").load(silver_paths["telemetry"])
trip_plans_df = spark.read.format("delta").load(silver_paths["trip_plans"])
vehicles_df = spark.read.format("delta").load(silver_paths["vehicles"])
drivers_df = spark.read.format("delta").load(silver_paths["drivers"])
routes_df = spark.read.format("delta").load(silver_paths["routes"])
shipments_df = spark.read.format("delta").load(silver_paths["shipments"])

quality_results = []

# ------------------------------------------------------------------------------
# 4. Critical Checks on Telemetry Events
# ------------------------------------------------------------------------------

# Check 4.1: Ensure mandatory identifiers and timestamps are not null
null_ids_count = telemetry_df.filter(
    F.col("event_id").isNull() | 
    F.col("vehicle_id").isNull() | 
    F.col("event_timestamp").isNull()
).count()
quality_results.append((
    run_id, 
    "telemetry_null_mandatory_ids", 
    "CRITICAL", 
    null_ids_count, 
    0, 
    "PASS" if null_ids_count == 0 else "FAIL"
))

# Check 4.2: Enforce primary key uniqueness (no duplicate event_id)
duplicate_events = telemetry_df.groupBy("event_id").count().filter("count > 1").count()
quality_results.append((
    run_id, 
    "telemetry_duplicate_event_ids", 
    "CRITICAL", 
    duplicate_events, 
    0, 
    "PASS" if duplicate_events == 0 else "FAIL"
))

# Check 4.3: Validate GPS spatial coordinates boundaries
invalid_gps_count = telemetry_df.filter(
    (F.col("latitude") < -90) | (F.col("latitude") > 90) |
    (F.col("longitude") < -180) | (F.col("longitude") > 180)
).count()
quality_results.append((
    run_id, 
    "telemetry_invalid_gps_coordinates", 
    "CRITICAL", 
    invalid_gps_count, 
    0, 
    "PASS" if invalid_gps_count == 0 else "FAIL"
))

# Check 4.4: Ensure numeric vehicle metrics are non-negative
negative_metrics_count = telemetry_df.filter(
    (F.col("speed_kph") < 0) | 
    (F.col("fuel_rate_lph") < 0) | 
    (F.col("rpm") < 0) | 
    (F.col("odometer_km") < 0)
).count()
quality_results.append((
    run_id, 
    "telemetry_negative_metric_values", 
    "CRITICAL", 
    negative_metrics_count, 
    0, 
    "PASS" if negative_metrics_count == 0 else "FAIL"
))

# ------------------------------------------------------------------------------
# 5. Referential Integrity Validation (Trip Plans vs Master Entities)
# ------------------------------------------------------------------------------
invalid_vehicle_refs = trip_plans_df.join(vehicles_df.select("vehicle_id").distinct(), "vehicle_id", "left_anti").count()
invalid_driver_refs = trip_plans_df.join(drivers_df.select("driver_id").distinct(), "driver_id", "left_anti").count()
invalid_route_refs = trip_plans_df.join(routes_df.select("route_id").distinct(), "route_id", "left_anti").count()
invalid_shipment_refs = trip_plans_df.join(shipments_df.select("shipment_id").distinct(), "shipment_id", "left_anti").count()

total_invalid_refs = invalid_vehicle_refs + invalid_driver_refs + invalid_route_refs + invalid_shipment_refs
quality_results.append((
    run_id, 
    "trip_plans_referential_integrity", 
    "CRITICAL", 
    total_invalid_refs, 
    0, 
    "PASS" if total_invalid_refs == 0 else "FAIL"
))

# ------------------------------------------------------------------------------
# 6. Audit Logging into Delta Lake Table (audit.data_quality_results)
# ------------------------------------------------------------------------------
schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("check_severity", StringType(), False),
    StructField("affected_records", LongType(), False),
    StructField("threshold", LongType(), False),
    StructField("status", StringType(), False)
])

results_df = spark.createDataFrame(quality_results, schema).withColumn("checked_at", F.current_timestamp())

# Persist quality results to the audit Delta table
results_df.write.format("delta").mode("append").save(audit_path)

print("=== QUALITY AUDIT SUMMARY ===")
results_df.show(truncate=False)

# ------------------------------------------------------------------------------
# 7. Conditional Gate Decision (Halt Pipeline on Critical Failure)
# ------------------------------------------------------------------------------
critical_failures = results_df.filter((F.col("check_severity") == "CRITICAL") & (F.col("status") == "FAIL")).count()

if critical_failures > 0:
    error_msg = f"CRITICAL QUALITY GATE FAILED: {critical_failures} critical checks failed! Halting pipeline to protect Gold layer."
    print(error_msg)
    raise Exception(error_msg)
else:
    print("ALL CRITICAL QUALITY CHECKS PASSED. Proceeding to Gold aggregation.")